## (Tutorial) Image Gradients
The difference in value between neighboring pixels is called an image gradient.<br>
We'll learn how to compute the image gradients with the following codes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.feature import hog
from skimage import data, color, exposure
import cv2

In [ ]:
# load image and turn it into grayscale
image = color.rgb2gray(data.retina())

In [ ]:
### Compute the horizontal gradient using the centered 1D filter.
### This is equivalent to replacing each non-border pixel with the
### difference between its right and left neighbors. The leftmost
### and rightmost edges have a gradient of 0.
gx = np.empty(image.shape, dtype=np.double)
gx[:, 0] = 0
gx[:, -1] = 0
gx[:, 1:-1] = image[:, :-2] - image[:, 2:]

In [ ]:
### Same deal for the vertical gradient
gy = np.empty(image.shape, dtype=np.double)
gy[0, :] = 0
gy[-1, :] = 0
gy[1:-1, :] = image[:-2, :] - image[2:, :]

In [ ]:
### Matplotlib incantations
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(5, 9), sharex=True, sharey=True)
ax1.axis('off')
ax1.imshow(image, cmap=plt.cm.gray)
ax1.set_title('Original image')
ax1.set_adjustable('box')
ax2.axis('off')
ax2.imshow(gx, cmap=plt.cm.gray)
ax2.set_title('Horizontal gradients')
ax2.set_adjustable('box')
ax3.axis('off')
ax3.imshow(gy, cmap=plt.cm.gray)
ax3.set_title('Vertical gradients')
ax3.set_adjustable('box')

## Task 1. Design your code to compute diagonal gradient of the image used in the example and visualize it

In [ ]:
# write your code here
# load image and turn it into grayscale
image1 = color.rgb2gray(data.retina())

#Horizontal Gradient
gx = np.empty(image1.shape, dtype=np.double)
gx[:, 0] = 0
gx[:, -1] = 0
gx[:, 1:-1] = image1[:, :-2] - image1[:, 2:]

# vertical gradient
gy = np.empty(image1.shape, dtype=np.double)
gy[0, :] = 0
gy[-1, :] = 0
gy[1:-1, :] = image1[:-2, :] - image1[2:, :]

#Diaonal gradient
gd = np.empty(image1.shape, dtype=np.double)
gd[:, 0] = 0
gd[:, -1]= 0
gd[0, :]= 0
gd[-1, :] = 0
gd[1:-1, 1:-1] = image1[:-2, :-2] - image1[2:, 2:]

### Matplotlib incantations
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(5, 12), sharex=True, sharey=True)
ax1.axis('off')
ax1.imshow(image1, cmap=plt.cm.gray)
ax1.set_title('Original image')
ax1.set_adjustable('box')
ax2.axis('off')
ax2.imshow(gx, cmap=plt.cm.gray)
ax2.set_title('Horizontal gradients')
ax2.set_adjustable('box')
ax3.axis('off')
ax3.imshow(gy, cmap=plt.cm.gray)
ax3.set_title('Vertical gradients')
ax3.set_adjustable('box')
ax4.axis('off')
ax4.imshow(gd, cmap=plt.cm.gray)
ax4.set_title('Diagonal gradients')
ax4.set_adjustable('box')

## **Question 1**. Explain why we don't directly use the simplest image features (color or number of pixels)

**Answer to Q1**: Simple image features, like raw pixel color values or the total number of pixels, aren't useful for complex tasks like image recognition. They don’t provide helpful information about the content or structure of an image. Changes in lighting, scale, rotation, or even slight shifts in position can significantly change these raw values. This makes it hard to consistently identify objects or patterns. More effective features, such as gradients, HOG, or ORB, are needed. These features are made to be less sensitive to variations and better at capturing local patterns and textures that indicate object identity.

---

## (Tutorial) HOG image features

In [ ]:
# read image and convert it to grayscale
image = color.rgb2gray(data.chelsea())
# extract HOG features from the input image with predefined parameters
fd, hog_image = hog(image, orientations=8, pixels_per_cell=(16, 16),
                    cells_per_block=(1, 1), visualize=True)

hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 0.02))

In [ ]:
# visualize the original image and the hog features
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4), sharex=True, sharey=True)

ax1.axis('off')
ax1.imshow(image, cmap=plt.cm.gray)
ax1.set_title('Input image')
ax1.set_adjustable('box')

ax2.axis('off')
ax2.imshow(hog_image_rescaled, cmap=plt.cm.gray)
ax2.set_title('Histogram of Oriented Gradients')
ax2.set_adjustable('box')

## Task 2. ORB keypoints extraction and visualization
Apart from HOG image features, another popular image feature is ORB.<br>
Please extract keypoints from the image used in HOG example using on ORB features and visualize the keypoints on the original image

In [ ]:
# write your code here
from matplotlib.transforms import offset_copy
import cv2
import matplotlib.pyplot as plt42

original_image = data.chelsea()
gray_image = cv2.cvtColor(original_image, cv2.COLOR_RGB2GRAY)

orb = cv2.ORB_create()
keypoints= orb.detect(gray_image, None)
keypoints_image = cv2.drawKeypoints(original_image, keypoints, None, color=(0, 255, 0), flags=0)

# Display the original image with keypoints
plt42.figure(figsize=(8, 6))
plt42.subplot(121)
plt42.axis('off')
plt42.imshow(original_image)
plt42.title('Original Image')
plt42.subplot(122)
plt42.axis('off')
plt42.imshow(cv2.cvtColor(keypoints_image, cv2.COLOR_BGR2RGB))
plt42.title('Keypoints Detected with ORB')
plt42.show()


## (Tutorial) Bag-of-Visual-Words
In the tutorial code, we create the code words only based on one image

In [ ]:
# defining feature extractor that we want to use (ORB feature used in this extractor)
def features(image):
    extractor = cv2.ORB_create()
    keypoints, descriptors = extractor.detectAndCompute(image, None)
    return keypoints, descriptors

In [ ]:
# builing visual words histogram based on the descriptors of the image and the clustering algorithm
def build_histogram(descriptor_list, cluster_alg):
    histogram = np.zeros(len(cluster_alg.cluster_centers_))
    cluster_result =  cluster_alg.predict(descriptor_list)
    for i in cluster_result:
        histogram[i] += 1.0
    return histogram

In [ ]:
# Clustring the bag-of-visual-words by Kmeans method
from sklearn.cluster import KMeans
img = data.chelsea()
kmeans = KMeans(n_clusters = 30)
kp, descriptor_list = features(img)
kmeans.fit(descriptor_list)
images = []
images.append(img)
preprocessed_image = []
for image in images:
      # image = gray(image)
      keypoint, descriptor = features(image)
      if (descriptor is not None):
          histogram = build_histogram(descriptor, kmeans)
          preprocessed_image.append(histogram)

In [ ]:
import pandas as pd
df = pd.DataFrame(preprocessed_image[0])
df.plot.bar()
#plt.grid(axis = 'y', alpha = 0.75)
plt.xlabel('Visual words index')
plt.ylabel('number')
plt.title('The histogram of visual words')

## Task 3. Bag-of-Visual-Words for multiple images
Create your bag-of-visual-words with 10 images and visualize the histogram of visual words. The 10 images belong to one category(dog, cat, bike, etc.)

In [ ]:
# write your code here
import cv2
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

#Extract Features
def extract_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Error loading image: {image_path}")
        return None
    # Check image depth - SIFT requires CV_8U
    if img.dtype != np.uint8:
        print(f"Image has incorrect depth: {image_path} (dtype: {img.dtype}). Expected uint8.")
        # Attempt to convert if possible, or return None
        try:
            img = cv2.convertScaleAbs(img) # Convert to 8-bit
        except Exception as e:
            print(f"Could not convert image depth for {image_path}: {e}")
            return None

    sift = cv2.SIFT_create()
    _, descriptors = sift.detectAndCompute(img, None)
    return descriptors

# --- Define paths to your images ---
# Assuming your images are in the /content/sample_data directory
image_folder = '/content/sample_data'
image_filenames = ["img1.jpeg", "img2.jpeg", "img3.jpeg", "img4.jpeg", "img5.jpeg",
                   "img6.jpeg", "img7.jpeg", "img8.jpeg", "img9.jpeg", "img10.jpeg"]
image_paths = [f"{image_folder}/{filename}" for filename in image_filenames]
# --- End of paths definition ---


all_features = []
for path in image_paths:
    descriptors = extract_features(path)
    if descriptors is not None:
        all_features.append(descriptors)

if not all_features:
    print("No features extracted from any images. Please check image paths and file types.")
else:
    # Concatenate descriptors from all images
    all_features = np.vstack(all_features).astype(np.float32)

    # --- Bag-of-Visual-Words Clustering ---
    k = 100 # Number of visual words (clusters)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # Added random_state and n_init for reproducibility
    kmeans.fit(all_features)
    # --- End of Clustering ---


    # --- Build Histograms of Visual Words for Each Image ---
    histograms = []
    for path in image_paths:
        img_descriptors = extract_features(path)
        if img_descriptors is not None:
            # Predict cluster for each descriptor in the image
            assignments = kmeans.predict(img_descriptors)
            # Build histogram
            histogram = np.zeros(k)
            for i in assignments:
                histogram[i] += 1
            histograms.append(histogram)
        else:
            # Append a zero histogram if no descriptors were extracted for an image
            histograms.append(np.zeros(k))

    # Normalize histograms (optional, but often useful)
    # Sum of all bins equals 1
    normalized_histograms = [hist / np.sum(hist) if np.sum(hist) > 0 else hist for hist in histograms]
    # --- End of Histogram Building ---


    # --- Visualize Histogram of Visual Words for the First Image ---
    if normalized_histograms:
        plt.figure(figsize=(12, 6))
        # We visualize the histogram of the first image in the list
        plt.bar(range(k), normalized_histograms[0])
        plt.xlabel('Visual Word Index')
        plt.ylabel('Normalized Frequency')
        plt.title('Histogram of Visual Words for the First Image')
        plt.show()
    else:
        print("No histograms could be generated.")

## Question 2. Recall that the feature scaling does not have a large effect on classification accuracy (tf-idf). How about the effect of feature scaling in Bag-of-Visual-Words? Explain it.

**Answer to Q2**: In Bag-of-Visual-Words, scaling features can greatly influence the results. In contrast, TF-IDF mainly affects how terms are weighted. BoVW uses clustering algorithms like K-Means to group similar local image features, which are called descriptors, into visual words. Clustering algorithms usually respond to the scale of the features. Features with larger values can dominate the distance calculations and cause biased clustering. Scaling features to a similar range, for instance through standardization or normalization, helps ensure that all features contribute more equally to the clustering process. This can lead to a better representation of visual words and may improve classification accuracy.

---

## Question 3. What is the advantage of ORB(Oriented FAST and Rotated Brief) over SURF (Speeded up robust features). Explain briefly (Minimum of 10 words and maximum of 150 words)

**Answer to Q3**: The primary benefit of ORB when compared to SURF is that it is free, while SURF is patented. This makes ORB a better financial choice for many computer vision applications, particularly commercial projects. SURF performs well and is fast, but ORB performs similarly in many situations, and does not require any licensing fees. ORB is also designed to be faster than SURF in some implementations.

## Task 4. Extract text from image
Image file is given in canvas assignment .

Save it in drive and use get shareable link to everyone option . Image file name is **ExtractText4.jpeg**

Please extract the image from image file and print the text that you extracted.

In [ ]:
# write your code here
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pytesseract
!sudo apt install tesseract-ocr

In [ ]:
from PIL import Image
import pytesseract
import os

# Path to the image file in Google Drive
# Using the path provided by the user - please verify this path in your Google Drive
image_path = '/content/drive/MyDrive/content/Task4_Image.jpeg'

# Check if the file exists at the path
if not os.path.exists(image_path):
    print(f"Error: Image file not found at {image_path}")
else:
    # Load the image
    image = Image.open(image_path)

    # Extract text from the image
    extracted_text = pytesseract.image_to_string(image)

    # Print the extracted text
    print(extracted_text)

## Task 5. Design code to implement **ORB** using **OpenCV**. Please use a face image (Preferably your face image as its unique).

Keep your face image file name as **faceimg.jpg**
Save it in drive and use get shareable link to everyone option.

Please print the following during the implementation

* Training image and Testing image

* Number of Keypoints Detected In The Training Image (Not just number image too)

* Number of Keypoints Detected In The Query Image (Not just number image too)

* Number of Matching Keypoints Between The Training and Query Images (Not just number image too)



In [ ]:
from google.colab import drive
import cv2
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# Image paths
img_path_train = '/content/drive/MyDrive/content/PXL_20250828_083754966.RAW-01.COVER.jpg'
img_path_query = '/content/drive/MyDrive/content/PXL_20250925_040912393.RAW-01.MP.COVER.jpg'

# Load images in grayscale
img_train = cv2.imread(img_path_train, cv2.IMREAD_GRAYSCALE)
img_query = cv2.imread(img_path_query, cv2.IMREAD_GRAYSCALE)

# Check if images loaded
if img_train is None or img_query is None:
    raise FileNotFoundError("Check image paths or file format.")

# ORB detector
orb = cv2.ORB_create()
kp_train, des_train = orb.detectAndCompute(img_train, None)
kp_query, des_query = orb.detectAndCompute(img_query, None)

# Draw keypoints
img_train_kp = cv2.drawKeypoints(img_train, kp_train, None, color=(0, 255, 0))
img_query_kp = cv2.drawKeypoints(img_query, kp_query, None, color=(0, 255, 0))

# Show keypoints
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img_train_kp, cmap='gray')
plt.title(f'Training Image with {len(kp_train)} Keypoints')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_query_kp, cmap='gray')
plt.title(f'Query Image with {len(kp_query)} Keypoints')
plt.axis('off')
plt.show()

# Match descriptors
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des_train, des_query)
matches = sorted(matches, key=lambda x: x.distance)

# Draw top 20 matches
img_matches = cv2.drawMatches(img_train, kp_train, img_query, kp_query, matches[:20], None, flags=2)

# Show matches
print(f"Number of Matching Keypoints: {len(matches)}")
plt.figure(figsize=(12, 6))
plt.imshow(img_matches)
plt.title('Top 20 Matches')
plt.axis('off')
plt.show()